# ====================================================================
#  CodeAlpha Internship—Task 1: Language Translation Tool
#  Developer Name: Tauseef Alam
#  Platform  : Google Colab
#  Libraries : deep_translator, gradio, langdetect
## ======================================================================

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Install Required Libraries
# Run this cell first. It installs everything you need.
# ─────────────────────────────────────────────────────────────

!pip install deep_translator gradio langdetect


# ─────────────────────────────────────────────────────────────
# CELL 2 — Import Libraries
# We import the tools we just installed so we can use them.
# ─────────────────────────────────────────────────────────────


from deep_translator import GoogleTranslator   # Handles translation via Google
from langdetect import detect, LangDetectException  # Detects what language text is in
import gradio as gr                            # Builds the interactive web UI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=e7cb93dc05f142f9022933a6cf1c670583a1e676e9dadc861a299ab27b84d08b
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Language List
# A dictionary mapping readable language names → their codes.
# Google Translate uses short codes like "en", "fr", "ur", etc.
# ─────────────────────────────────────────────────────────────

LANGUAGES = {
    "Auto Detect"        : "auto",
    "English"            : "en",
    "Urdu"               : "ur",
    "Arabic"             : "ar",
    "Hindi"              : "hi",
    "French"             : "fr",
    "Spanish"            : "es",
    "German"             : "de",
    "Chinese (Simplified)": "zh-CN",
    "Japanese"           : "ja",
    "Russian"            : "ru",
    "Portuguese"         : "pt",
    "Italian"            : "it",
    "Korean"             : "ko",
    "Turkish"            : "tr",
    "Dutch"              : "nl",
    "Polish"             : "pl",
    "Bengali"            : "bn",
    "Persian (Farsi)"    : "fa",
    "Punjabi"            : "pa",
    "Pashto"             : "ps",
    "Swahili"            : "sw",
    "Indonesian"         : "id",
}

# Get just the names for the dropdown menus in the UI
lang_names = list(LANGUAGES.keys())


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Auto Language Detector Helper
# This function figures out what language the user typed in,
# so they don't have to manually select "Auto Detect".
# ─────────────────────────────────────────────────────────────

def detect_language(text):
    """
    Tries to detect the language of the input text.
    Returns a language code like 'en', 'ur', 'ar', etc.
    Falls back to 'en' if detection fails.
    """
    try:
        if not text or len(text.strip()) < 3:
            return "en"                    # Too short to detect — assume English
        detected_code = detect(text)       # e.g. returns "ur" for Urdu text
        return detected_code
    except LangDetectException:
        return "en"                        # If something goes wrong, default to English


In [4]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Core Translation Function
# This is the heart of the project.
# It takes the user's text + language choices → returns translation.
# ─────────────────────────────────────────────────────────────

def translate_text(text, source_lang_name, target_lang_name):
    """
    Translates text from source language to target language.

    Parameters:
        text             : The text the user wants to translate
        source_lang_name : e.g. "Auto Detect", "English", "Urdu"
        target_lang_name : e.g. "French", "Arabic", "Spanish"

    Returns:
        A string — either the translated text or an error message.
    """

    # ── Step 1: Check that the user actually typed something ──
    if not text or not text.strip():
        return "⚠️ Please enter some text to translate."

    # ── Step 2: Enforce a character limit (Google Translate has a limit) ──
    if len(text) > 5000:
        return "⚠️ Text is too long. Please keep it under 5000 characters."

    # ── Step 3: Convert language names → codes ──
    source_code = LANGUAGES[source_lang_name]   # e.g. "auto" or "en"
    target_code = LANGUAGES[target_lang_name]   # e.g. "fr"

    # ── Step 4: If source == target, no translation needed ──
    if source_code == target_code and source_code != "auto":
        return "⚠️ Source and target languages are the same. Please choose different languages."

    # ── Step 5: If auto-detect is selected, detect the language first ──
    if source_code == "auto":
        source_code = detect_language(text)

    # ── Step 6: Perform the actual translation ──
    try:
        translator = GoogleTranslator(source=source_code, target=target_code)
        translated = translator.translate(text)

        if translated is None:
            return "❌ Translation returned empty. Please try again."

        return translated

    except Exception as error:
        # Show a helpful error message if something goes wrong
        return f"❌ Translation failed: {str(error)}\n\nTip: Check your internet connection."

In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Character Counter Helper
# Updates the "X / 5000 characters" counter below the text box.
# ─────────────────────────────────────────────────────────────

def update_char_count(text):
    """Returns a string showing how many characters have been typed."""
    count = len(text) if text else 0
    color_warning = " ⚠️ (too long!)" if count > 5000 else ""
    return f"{count} / 5000 characters{color_warning}"


In [6]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Swap Languages Helper
# Lets the user flip source and target language in one click.
# Also moves the translated text back to input for re-translation.
# ─────────────────────────────────────────────────────────────

def swap_languages(src, tgt, translated_text):
    """
    Swaps source and target language selections.
    Also puts the translated text into the input box.
    """
    # If source was "Auto Detect", swap to English instead
    new_src = tgt
    new_tgt = src if src != "Auto Detect" else "English"
    new_input = translated_text if translated_text else ""
    return new_src, new_tgt, new_input, ""   # Clear the output box

In [7]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — Clear All Function
# Resets everything back to empty / default state.
# ─────────────────────────────────────────────────────────────

def clear_all():
    """Clears input, output, and resets the character counter."""
    return "", "", "0 / 5000 characters"


In [8]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — Build the Gradio UI
# This creates the full interactive web interface.
# Gradio turns Python functions into beautiful web apps!
# ─────────────────────────────────────────────────────────────

def build_app():
    """Builds and returns the complete Gradio application."""

    with gr.Blocks(
        title="Language Translation Tool — CodeAlpha",
        theme=gr.themes.Soft(
            primary_hue="blue",
            secondary_hue="indigo",
        )
    ) as demo:

        # ── Header Section ──────────────────────────────────
        gr.Markdown("""
        # 🌍 Language Translation Tool
        **CodeAlpha AI Internship — Task 1**
        Translate text across 22+ languages instantly using Google Translate.
        """)

        # ── Main Translation Area ────────────────────────────
        with gr.Row():

            # Left Column — Input Side
            with gr.Column(scale=1):

                source_lang = gr.Dropdown(
                    choices=lang_names,
                    value="Auto Detect",       # Default: auto-detect input language
                    label="📥 Source Language",
                    interactive=True
                )

                input_text = gr.Textbox(
                    lines=10,
                    placeholder="Type or paste your text here...\n\nExample: Hello, how are you?",
                    label="Input Text",
                    max_lines=20
                )

                char_count = gr.Textbox(
                    value="0 / 5000 characters",
                    label="",
                    interactive=False,
                    container=False,
                    elem_id="char-counter"
                )

            # Middle Column — Action Buttons
            with gr.Column(scale=0, min_width=80):
                gr.Markdown("<br><br><br><br><br>")   # spacing
                swap_btn = gr.Button("⇄", size="sm", variant="secondary")

            # Right Column — Output Side
            with gr.Column(scale=1):

                target_lang = gr.Dropdown(
                    choices=lang_names,
                    value="Urdu",              # Default target: Urdu (good for Pakistani students)
                    label="📤 Target Language",
                    interactive=True
                )

                output_text = gr.Textbox(
                    lines=10,
                    label="Translated Text",
                    max_lines=20,
                    interactive=False,
                    show_copy_button=True      # Built-in copy button — task requirement!
                )

        # ── Action Buttons Row ───────────────────────────────
        with gr.Row():
            translate_btn = gr.Button(
                "🔄 Translate",
                variant="primary",
                size="lg",
                scale=3
            )
            clear_btn = gr.Button(
                "🗑️ Clear All",
                variant="secondary",
                size="lg",
                scale=1
            )

        # ── Example Inputs ───────────────────────────────────
        gr.Markdown("### 💡 Try These Examples")
        gr.Examples(
            examples=[
                ["Hello! How are you doing today?",        "Auto Detect", "Urdu"],
                ["I am learning Artificial Intelligence.", "Auto Detect", "Arabic"],
                ["The weather is beautiful today.",        "Auto Detect", "French"],
                ["میں پاکستان سے ہوں اور AI سیکھ رہا ہوں۔", "Auto Detect", "English"],
                ["La vie est belle.",                      "Auto Detect", "Spanish"],
                ["こんにちは、元気ですか？",                  "Auto Detect", "English"],
            ],
            inputs=[input_text, source_lang, target_lang],
            label="Click any example to load it"
        )

        # ── How It Works Section ─────────────────────────────
        with gr.Accordion("ℹ️ How does this work?", open=False):
            gr.Markdown("""
            **Step-by-step process:**
            1. You type text in the **Input Text** box
            2. Select **Source Language** (or leave on Auto Detect — it will figure it out!)
            3. Select the **Target Language** you want to translate to
            4. Click **Translate** — the tool calls Google Translate API
            5. The result appears on the right with a **Copy** button

            **Libraries used:**
            - `deep_translator` → Sends your text to Google Translate for free
            - `langdetect` → Detects what language your text is written in
            - `gradio` → Builds this interactive web interface
            """)

        # ─────────────────────────────────────────────────────
        # WIRE UP EVENTS
        # This connects buttons/inputs to the Python functions above.
        # ─────────────────────────────────────────────────────

        # When user clicks "Translate" button
        translate_btn.click(
            fn=translate_text,
            inputs=[input_text, source_lang, target_lang],
            outputs=output_text
        )

        # Allow pressing Enter to also translate (via submit on textbox)
        input_text.submit(
            fn=translate_text,
            inputs=[input_text, source_lang, target_lang],
            outputs=output_text
        )

        # Update character counter as user types
        input_text.change(
            fn=update_char_count,
            inputs=input_text,
            outputs=char_count
        )

        # Swap languages button
        swap_btn.click(
            fn=swap_languages,
            inputs=[source_lang, target_lang, output_text],
            outputs=[source_lang, target_lang, input_text, output_text]
        )

        # Clear everything button
        clear_btn.click(
            fn=clear_all,
            outputs=[input_text, output_text, char_count]
        )

    return demo

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — Launch the App
# share=True gives you a public URL you can open on any device!
# This URL lasts for 72 hours — perfect for recording your video.
# ─────────────────────────────────────────────────────────────

app = build_app()
app.launch(
    share=True,       # Creates a public link like: https://xxxxx.gradio.live
    debug=True        # Shows error details in Colab if something breaks
)


/tmp/ipykernel_3867/2870341391.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6eef1bf59cf20e14a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
